<a href="https://colab.research.google.com/github/josuemarroquinj/corporate-value-portfolio-risk-analysis/blob/main/notebooks/00_fundamental_financial_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import os

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
!pip install openpyxl

In [3]:
import pandas as pd

# URL RAW del archivo en tu repositorio de GitHub
excel_url = 'https://raw.githubusercontent.com/josuemarroquinj/corporate-value-portfolio-risk-analysis/main/data/raw/Financials2.xlsx'

# Pandas puede leer directamente enlaces a archivos Excel alojados en GitHub
xls = pd.ExcelFile(excel_url)
print("Hojas encontradas:", xls.sheet_names)

Hojas encontradas: ['Financials_NVDA', 'Finacials_MSFT', 'Financials_AMZN', 'Financials_BRK.B', 'Financials_XOM']


In [4]:
import pandas as pd
import numpy as np

def process_company_financials(excel_file_obj, tickers_betas, rf_rate=0.042, erp=0.055):
    """
    Procesa las hojas de estados financieros de un objeto pd.ExcelFile
    y calcula NOPAT, WACC, Beta Desapalancado (Hamada) y EVA.

    Parámetros:
    - excel_file_obj: Objeto pd.ExcelFile cargado desde GitHub
    - tickers_betas: Diccionario con los Betas apalancados de cada empresa (Fase 2)
    - rf_rate: Tasa libre de riesgo (default 4.2% -> Bono del Tesoro EE.UU. a 10 años)
    - erp: Prima de riesgo de mercado / Equity Risk Premium (default 5.5%)
    """
    results = []

    for sheet in excel_file_obj.sheet_names:
        ticker = sheet.strip().upper()

        # Leer la hoja de la empresa
        df = pd.read_excel(excel_file_obj, sheet_name=sheet)

        # Función auxiliar para extracción flexible de filas independientemente del nombre exacto
        def get_val(pattern):
            col_item = df.columns[0]
            col_val = df.columns[1]
            mask = df[col_item].astype(str).str.contains(pattern, case=False, na=False)
            vals = df.loc[mask, col_val].values
            return float(vals[0]) if len(vals) > 0 else 0.0

        # 1. Extraer rubros del Estado de Resultados
        ebit = get_val('EBIT|Operating Income|Ingreso Operativo')
        tax_expense = get_val('Tax Expense|Impuestos|Income Tax')
        ebt = get_val('EBT|Pretax Income|Utilidad Antes de Impuestos')
        interest_expense = get_val('Interest Expense|Gastos Financieros|Intereses')

        # 2. Extraer rubros del Balance General
        total_debt = get_val('Total Debt|Deuda Total')
        total_equity = get_val('Total Equity|Stockholders Equity|Patrimonio')
        cash = get_val('Cash and Cash Equivalents|Efectivo')

        # 3. Cálculo de la Tasa Impositiva Efectiva y NOPAT
        if ebt > 0 and tax_expense > 0:
            effective_tax_rate = tax_expense / ebt
            # Acotar la tasa entre 0% y 35% para evitar sesgos por ítems extraordinarios
            effective_tax_rate = max(0.0, min(effective_tax_rate, 0.35))
        else:
            effective_tax_rate = 0.21  # Tasa corporativa estándar en EE.UU.

        nopat = ebit * (1 - effective_tax_rate)

        # 4. Capital Invertido (IC)
        invested_capital = total_debt + total_equity - cash

        # 5. Estructura de Capital y Ratios
        total_val = total_debt + total_equity
        w_d = total_debt / total_val if total_val > 0 else 0.0
        w_e = total_equity / total_val if total_val > 0 else 1.0
        de_ratio = total_debt / total_equity if total_equity > 0 else 0.0

        # 6. Costo de Capital Propio (Ke) y Costo de la Deuda (Kd)
        beta_l = tickers_betas.get(ticker, 1.0)
        ke = rf_rate + (beta_l * erp)

        kd_before_tax = interest_expense / total_debt if total_debt > 0 else 0.04
        kd_after_tax = kd_before_tax * (1 - effective_tax_rate)

        # 7. Desapalancamiento del Beta (Ecuación de Hamada)
        beta_u = beta_l / (1 + (1 - effective_tax_rate) * de_ratio)

        # 8. WACC, ROIC y EVA
        wacc = (w_e * ke) + (w_d * kd_after_tax)
        roic = nopat / invested_capital if invested_capital > 0 else 0.0
        eva = nopat - (invested_capital * wacc)
        eva_spread = roic - wacc

        # Consolidar dataframe por empresa
        results.append({
            'Ticker': ticker,
            'EBIT': ebit,
            'Tax_Rate_Eff': effective_tax_rate,
            'NOPAT': nopat,
            'Invested_Capital': invested_capital,
            'ROIC': roic,
            'Levered_Beta (βL)': beta_l,
            'Unlevered_Beta_Hamada (βU)': beta_u,
            'Cost_of_Equity (Ke)': ke,
            'Cost_of_Debt_AfterTax (Kd)': kd_after_tax,
            'WACC': wacc,
            'EVA': eva,
            'EVA_Spread (ROIC - WACC)': eva_spread
        })

    return pd.DataFrame(results)

print("Función 'process_company_financials' cargada correctamente en Colab.")

Función 'process_company_financials' cargada correctamente en Colab.


In [8]:
import pandas as pd
import numpy as np
import re

def process_company_financials(excel_file_obj, tickers_betas, rf_rate=0.042, erp=0.055):
    """
    Procesa las hojas de estados financieros extrayendo dinámicamente el ticker,
    calculando NOPAT, WACC, Beta Desapalancado (Hamada) y EVA.
    """
    results = []

    # Limpiador numérico para texto con formato moneda/comas/paréntesis
    def clean_val(val):
        if pd.isna(val):
            return 0.0
        if isinstance(val, (int, float)):
            return float(val)
        val_str = str(val).strip().replace('$', '').replace(',', '').replace(' ', '')
        if val_str.startswith('(') and val_str.endswith(')'):
            val_str = '-' + val_str[1:-1]
        try:
            return float(val_str)
        except ValueError:
            return 0.0

    for sheet in excel_file_obj.sheet_names:
        # Extraer únicamente el Ticker eliminando prefijos tipo FINANCIALS_ o FINACIALS_
        raw_sheet = sheet.strip().upper()
        clean_ticker = re.sub(r'^FINAN?CIALS_', '', raw_sheet)

        df = pd.read_excel(excel_file_obj, sheet_name=sheet)

        # Búsqueda flexible de rubros en la primera columna
        def get_val(pattern):
            col_item = df.columns[0]
            col_val = df.columns[1]
            mask = df[col_item].astype(str).str.contains(pattern, case=False, na=False)
            vals = df.loc[mask, col_val].values
            return clean_val(vals[0]) if len(vals) > 0 else 0.0

        # 1. Rubros del Estado de Resultados
        ebit = get_val('EBIT|Operating Income|Ingreso Operativo')
        tax_expense = get_val('Tax Expense|Impuestos|Income Tax')
        ebt = get_val('EBT|Pretax Income|Utilidad Antes de Impuestos')
        interest_expense = get_val('Interest Expense|Gastos Financieros|Intereses')

        # 2. Rubros del Balance General (patrones ampliados)
        total_debt = get_val('Total Debt|Deuda Total|Long Term Debt|Short Term Debt')
        total_equity = get_val('Total Equity|Stockholders Equity|Patrimonio|Total Stockholders')
        cash = get_val('Cash and Cash Equivalents|Efectivo|Cash')

        # 3. Tasa Impositiva Efectiva y NOPAT
        if ebt > 0 and tax_expense > 0:
            effective_tax_rate = max(0.0, min(tax_expense / ebt, 0.35))
        else:
            effective_tax_rate = 0.21

        nopat = ebit * (1 - effective_tax_rate)

        # 4. Capital Invertido
        invested_capital = total_debt + total_equity - cash if (total_debt + total_equity) > 0 else (ebit / 0.15)

        # 5. Estructura de Capital
        total_val = total_debt + total_equity
        w_d = total_debt / total_val if total_val > 0 else 0.0
        w_e = total_equity / total_val if total_val > 0 else 1.0
        de_ratio = total_debt / total_equity if total_equity > 0 else 0.0

        # 6. Betas y Costo de Capital (CAPM)
        beta_l = tickers_betas.get(clean_ticker, 1.0)
        ke = rf_rate + (beta_l * erp)

        kd_before_tax = interest_expense / total_debt if total_debt > 0 else 0.04
        kd_after_tax = kd_before_tax * (1 - effective_tax_rate)

        # 7. Desapalancamiento de Beta (Hamada)
        beta_u = beta_l / (1 + (1 - effective_tax_rate) * de_ratio)

        # 8. WACC y EVA
        wacc = (w_e * ke) + (w_d * kd_after_tax)
        roic = nopat / invested_capital if invested_capital > 0 else 0.0
        eva = nopat - (invested_capital * wacc)
        eva_spread = roic - wacc

        results.append({
            'Ticker': clean_ticker,
            'EBIT': ebit,
            'Tax_Rate_Eff': effective_tax_rate,
            'NOPAT': nopat,
            'Invested_Capital': invested_capital,
            'ROIC': roic,
            'Levered_Beta (βL)': beta_l,
            'Unlevered_Beta_Hamada (βU)': beta_u,
            'Cost_of_Equity (Ke)': ke,
            'Cost_of_Debt_AfterTax (Kd)': kd_after_tax,
            'WACC': wacc,
            'EVA': eva,
            'EVA_Spread (ROIC - WACC)': eva_spread
        })

    return pd.DataFrame(results)

print("Función 'process_company_financials' corregida y optimizada.")

Función 'process_company_financials' corregida y optimizada.


In [9]:
# Betas apalancados observados/estimados en la Fase 2
# Ajusta los tickers según los nombres exactos de las pestañas de tu Excel
betas_fase2 = {
    'NVDA': 1.72,
    'AMZN': 1.18,
    'MSFT': 0.89,
    'XOM': 0.55,
    'BRK.B': 0.82
}

# 1. Ejecutar la función sobre el archivo cargado en Celda 2
df_eva_results = process_company_financials(xls, betas_fase2)

# 2. Formatear la tabla para presentación visual
df_display = df_eva_results.copy()

for col in ['ROIC', 'Cost_of_Equity (Ke)', 'Cost_of_Debt_AfterTax (Kd)', 'WACC', 'EVA_Spread (ROIC - WACC)']:
    df_display[col] = df_display[col].map('{:.2%}'.format)

for col in ['EBIT', 'NOPAT', 'Invested_Capital', 'EVA']:
    df_display[col] = df_display[col].map('${:,.2f}'.format)

# 3. Mostrar la tabla de resultados en Colab
display(df_display)

,Ticker,EBIT,Tax_Rate_Eff,NOPAT,Invested_Capital,ROIC,Levered_Beta (βL),Unlevered_Beta_Hamada (βU),Cost_of_Equity (Ke),Cost_of_Debt_AfterTax (Kd),WACC,EVA,EVA_Spread (ROIC - WACC)
0,NVDA,"$130,387,000.00",0.35,"$84,751,550.00","$146,688,000.00",57.78%,1.72,1.72,13.66%,2.60%,13.66%,"$64,713,969.20",44.12%
1,MSFT,"$155,237,000.00",0.35,"$100,904,050.00","$421,452,000.00",23.94%,0.89,0.89,9.10%,2.60%,9.10%,"$62,572,990.60",14.85%
2,AMZN,"$79,975,000.00",0.21,"$63,180,250.00","$324,255,000.00",19.48%,1.18,1.18,10.69%,3.16%,10.69%,"$28,517,390.50",8.79%
3,BRK.B,"$92,049,000.00",0.21,"$72,718,710.00","$665,542,000.00",10.93%,0.82,0.82,8.71%,3.16%,8.71%,"$14,750,001.80",2.22%
4,XOM,"$33,538,000.00",0.35,"$21,799,700.00","$248,705,000.00",8.77%,0.55,0.55,7.23%,2.60%,7.23%,"$3,830,763.75",1.54%
